In [1]:
import os
import sys

PROJECT_ROOT = "/Users/karima/Ironhack-challenges/fake-news-nlp-classification"

os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

print(os.getcwd())


/Users/karima/Ironhack-challenges/fake-news-nlp-classification


In [2]:
# Run once if the libraries are not installed:
# %pip install transformers accelerate torch


In [3]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from src.data_loader import (
    load_data,
    prepare_features_and_labels,
    split_data
)
from src.transformers import (
    MODEL_NAME,
    combine_title_and_text,
    load_distilbert_tokenizer,
    create_distilbert_dataset
)
from src.evaluator import (
    save_confusion_matrix,
    save_metrics_plot
)
from src.experiment_tracker import save_experiment_result


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/Caskroom/miniconda/base/envs/fake-news-nlp/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/Caskroom/miniconda

In [4]:
MODEL_ID = "exp_17"
MODEL_NAME_DISPLAY = "DistilBERT"
FEATURES = "DistilBERT contextual embeddings"
PREPROCESSING = "Title + text combined; DistilBERT tokenizer"
ALGORITHM = "DistilBERT sequence classification"
NOTES = "Fine-tuned distilbert-base-uncased for fake news classification."

MAX_LENGTH = 256
EPOCHS = 2
BATCH_SIZE = 8
RANDOM_STATE = 42


In [5]:
# 1. Load data
data = load_data()

X, y = prepare_features_and_labels(data)

# Optional: use a sample while testing the notebook
USE_SAMPLE = True
SAMPLE_SIZE = 3000

if USE_SAMPLE:
    sample_indices = y.sample(
        n=min(SAMPLE_SIZE, len(y)),
        random_state=RANDOM_STATE
    ).index

    X = X.loc[sample_indices].reset_index(drop=True)
    y = y.loc[sample_indices].reset_index(drop=True)

print(X.shape)


(3000, 2)


In [6]:
# 2. Split train and test
X_train_full, X_test, y_train_full, y_test = split_data(X, y)

# Create a validation set from the training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (2040, 2)
Validation: (360, 2)
Test: (600, 2)


In [7]:
# 3. Combine title and text
X_train_combined = combine_title_and_text(X_train)
X_val_combined = combine_title_and_text(X_val)
X_test_combined = combine_title_and_text(X_test)

X_train_combined[["title", "text", "combined_text"]].head()


,title,text,combined_text
241,"Clinton, Sanders audition for role as anti-Tru...",MILWAUKEE (Reuters) - Democratic presidential ...,"Clinton, Sanders audition for role as anti-Tru..."
2048,Clinton says U.S. presidential election an 'al...,LOS ANGELES (Reuters) - Democrat Hillary Clint...,Clinton says U.S. presidential election an 'al...
2954,SCARAMUCCI’S WIFE FILES FOR DIVORCE…Why She’s ...,When was the last time being loyal to your bos...,SCARAMUCCI’S WIFE FILES FOR DIVORCE…Why She’s ...
902,Catalan parliament to meet on Thursday to deci...,MADRID (Reuters) - Catalonia s parliament will...,Catalan parliament to meet on Thursday to deci...
2550,Orlando Gunman’s Ties To Anti-Gay Muslim Extr...,"At around 2:00 am on Sunday, June 12, Omar Mat...",Orlando Gunman’s Ties To Anti-Gay Muslim Extre...


In [8]:
# 4. Load DistilBERT tokenizer
tokenizer = load_distilbert_tokenizer()


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
# 5. Create tokenized datasets
train_dataset = create_distilbert_dataset(
    X_train,
    y_train,
    tokenizer,
    max_length=MAX_LENGTH
)

val_dataset = create_distilbert_dataset(
    X_val,
    y_val,
    tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = create_distilbert_dataset(
    X_test,
    y_test,
    tokenizer,
    max_length=MAX_LENGTH
)


In [10]:
# 6. Load pretrained DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


ImportError: 
AutoModelForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
# 7. Define evaluation metrics
def compute_metrics(prediction_output):
    logits, labels = prediction_output
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0)
    }


In [ ]:
# 8. Configure training
training_args = TrainingArguments(
    output_dir="results/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)


In [ ]:
# 9. Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [ ]:
# 10. Fine-tune DistilBERT
trainer.train()


In [ ]:
# 11. Evaluate on the test set
test_output = trainer.predict(test_dataset)

test_predictions = np.argmax(
    test_output.predictions,
    axis=1
)

metrics = {
    "test_accuracy": accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
    "f1_score": f1_score(y_test, test_predictions, zero_division=0)
}

metrics


In [ ]:
# 12. Save plots
metrics_plot_path = save_metrics_plot(
    metrics,
    MODEL_ID
)

confusion_matrix_path = save_confusion_matrix(
    y_test,
    test_predictions,
    MODEL_ID
)


In [ ]:
# 13. Save model and tokenizer
MODEL_PATH = "models/exp_17_distilbert"

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print(f"Model and tokenizer saved to: {MODEL_PATH}")


In [ ]:
# 14. Track experiment
experiment = {
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME_DISPLAY,
    "features": FEATURES,
    "preprocessing": PREPROCESSING,
    "algorithm": ALGORITHM,
    "train_accuracy": None,
    "test_accuracy": metrics["test_accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1_score": metrics["f1_score"],
    "notes": NOTES,
    "model_path": MODEL_PATH
}

tracking = save_experiment_result(**experiment)
tracking.tail()
